## 06 — Call the predictive API (OVMS / KServe v2)

**Goal:** send images to the deployed model and **see** the prediction.

### Endpoints (validated on this cluster)

| Kind | URL |
|------|-----|
| **Internal (use this in workbench)** | `http://muffin-chihuahua-predictor.chihuahua-vs-muffin-jan.svc.cluster.local:8888` |
| External | `https://muffin-chihuahua-chihuahua-vs-muffin-jan.apps.ocp.dv6rj.sandbox1011.opentlc.com` |

**Why not `:8080`?** The dashboard shows `:8080`, but the Service is headless
(`ClusterIP: None`) with `port 80 → targetPort 8888`. The pod listens on **8888**.
Connecting to `:80` or `:8080` → Connection refused.


In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import requests
from PIL import Image

warnings.filterwarnings("ignore", message="Unverified HTTPS request")

MODEL = "muffin-chihuahua"
CLASSES = ["chihuahua", "muffin"]
NS = "chihuahua-vs-muffin-jan"

CANDIDATES = [
    # Prefer internal :8888 (headless Service → pod port; UI :8080 is wrong here)
    f"http://{MODEL}-predictor.{NS}.svc.cluster.local:8888",
    f"https://{MODEL}-{NS}.apps.ocp.dv6rj.sandbox1011.opentlc.com",
    f"http://{MODEL}-predictor.{NS}.svc.cluster.local:80",
    f"http://{MODEL}-predictor.{NS}.svc.cluster.local:8080",
]

EP = None
for base in CANDIDATES:
    url = f"{base}/v2/models/{MODEL}"
    try:
        r = requests.get(url, timeout=8, verify=False)
        print(f"TRY {url} → {r.status_code}")
        if r.status_code == 200:
            EP = base
            print(r.text[:400])
            break
    except Exception as e:
        print(f"TRY {url} → FAIL ({type(e).__name__})")

assert EP, "No working endpoint"
print("Using EP =", EP)

In [ ]:
def preprocess(path: Path) -> np.ndarray:
    """Same preprocess as training: 224×224, ImageNet normalize, NCHW."""
    img = Image.open(path).convert("RGB").resize((224, 224))
    arr = np.asarray(img).astype("float32") / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype="float32")
    std = np.array([0.229, 0.224, 0.225], dtype="float32")
    arr = (arr - mean) / std
    return np.transpose(arr, (2, 0, 1))[None, ...]


def predict(path: Path) -> dict:
    chw = preprocess(path)
    payload = {
        "inputs": [
            {
                "name": "input",
                "shape": list(chw.shape),
                "datatype": "FP32",
                "data": chw.flatten().tolist(),
            }
        ]
    }
    r = requests.post(
        f"{EP}/v2/models/{MODEL}/infer",
        headers={"Content-Type": "application/json"},
        data=json.dumps(payload),
        timeout=30,
        verify=False,
    )
    r.raise_for_status()
    out = np.array(r.json()["outputs"][0]["data"], dtype="float32")
    exp = np.exp(out - out.max())
    probs = exp / exp.sum()
    idx = int(probs.argmax())
    return {
        "path": path,
        "true": path.parent.name,  # folder name = ground truth
        "label": CLASSES[idx],
        "confidence": float(probs[idx]),
        "probs": {CLASSES[i]: float(probs[i]) for i in range(len(CLASSES))},
    }


def show_prediction(result: dict):
    """Visual result: image + bar chart of class probabilities."""
    img = Image.open(result["path"]).convert("RGB")
    ok = result["label"] == result["true"]
    title_color = "green" if ok else "red"

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    axes[0].imshow(img)
    axes[0].axis("off")
    axes[0].set_title(
        f"pred: {result['label']} ({result['confidence']:.1%})\n"
        f"true: {result['true']}",
        color=title_color,
        fontsize=12,
    )

    labels = list(result["probs"].keys())
    values = list(result["probs"].values())
    colors = ["#4C9AFF" if lb == result["label"] else "#D2D2D2" for lb in labels]
    axes[1].barh(labels, values, color=colors)
    axes[1].set_xlim(0, 1)
    axes[1].set_xlabel("probability")
    axes[1].set_title("Class scores")
    for i, v in enumerate(values):
        axes[1].text(v + 0.02, i, f"{v:.1%}", va="center")

    fig.tight_layout()
    plt.show()
    return result

In [ ]:
# --- One image (quick check) ---
one = next(Path("/opt/app-root/src/data/muffin-chihuahua/test/muffin").glob("*.jpg"))
show_prediction(predict(one))

In [ ]:
# --- Visual gallery: a few muffins + a few chihuahuas ---
root = Path("/opt/app-root/src/data/muffin-chihuahua/test")
samples = list(root.joinpath("muffin").glob("*.jpg"))[:3] + list(
    root.joinpath("chihuahua").glob("*.jpg")
)[:3]

results = []
for p in samples:
    results.append(show_prediction(predict(p)))

# Summary table
print("\n=== Summary ===")
correct = 0
for r in results:
    mark = "OK" if r["label"] == r["true"] else "MISS"
    correct += r["label"] == r["true"]
    print(f"{mark:4} true={r['true']:10} pred={r['label']:10} conf={r['confidence']:.1%}  {r['path'].name}")
print(f"Accuracy on sample: {correct}/{len(results)}")